# Day 4: Linear Regression — Student Practice Notebook

**Programme:** Educentriks – Applied Data Science, AI/ML  
**Dataset:** TabReD Delivery ETA  
**Environment:** Python 3.13.x, Jupyter Notebook, `.venv`

## Learning outcomes
1. Prepare the approved TabReD arrays and use the supplied default train/validation/test indices.
2. Train Linear Regression and Ridge Regression models using a scikit-learn pipeline.
3. Evaluate regression predictions using MAE, MSE, RMSE and R².
4. Compare model performance and communicate limitations responsibly.

**Dataset documentation note:** Available metadata confirms a regression task scored using RMSE. It does not document the target's business meaning, unit, transformation or the meaning of every encoded feature array. This notebook therefore does not label the target as minutes or hours. It uses `x_num` and `x_bin`; `x_cat` and `x_meta` remain excluded until their meanings are verified.

## 1. Setup and dataset location

Expected project-relative location:

```text
data/TabReD_Delivery_ETA/delivery-eta/
    info.json
    x_num.npy
    x_bin.npy
    x_cat.npy
    x_meta.npy
    y.npy
    splits/default/train.npy
    splits/default/val.npy
    splits/default/test.npy
```

Install dependencies in the project's Python 3.13 virtual environment if needed:

```powershell
python -m pip install numpy pandas matplotlib scikit-learn
```

Run from the repository root, or update `DATA_DIR`.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("data") / "TabReD_Delivery_ETA" / "delivery-eta"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATA_DIR.resolve()}. "
        "Check extraction location or update DATA_DIR."
    )

print("Dataset folder:", DATA_DIR.resolve())

## 2. Inspect the supplied metadata

Do not infer target meaning or units from numeric values alone.

In [ ]:
with (DATA_DIR / "info.json").open("r", encoding="utf-8") as file:
    dataset_info = json.load(file)

print(json.dumps(dataset_info, indent=4))

task_info = dataset_info.get("task", {})
assert task_info.get("type") == "regression"
assert task_info.get("score") == "rmse"

print("Confirmed task:", task_info["type"])
print("Documented benchmark metric:", task_info["score"])

## 3. Load the approved arrays

| Array | Shape | Dtype | Treatment |
|---|---:|---|---|
| `x_num.npy` | `(416451, 221)` | `float32` | Include |
| `x_bin.npy` | `(416451, 1)` | `float32` | Include |
| `x_cat.npy` | `(416451, 1)` | `int64` | Exclude pending encoding documentation |
| `x_meta.npy` | `(416451, 2)` | `int64` | Exclude pending meaning documentation |
| `y.npy` | `(416451,)` | `float32` | Target |

The baseline uses 222 input columns. Their business feature names are not established by the available material.

In [ ]:
X_num = np.load(DATA_DIR / "x_num.npy")
X_bin = np.load(DATA_DIR / "x_bin.npy")
y = np.load(DATA_DIR / "y.npy")

assert X_num.shape[0] == X_bin.shape[0] == y.shape[0]
X = np.concatenate([X_num, X_bin], axis=1)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X dtype:", X.dtype, "| y dtype:", y.dtype)

## 4. Inspect the data

Check missing values and the target distribution. The target's physical unit is not established by the available metadata.

In [ ]:
print("Records:", X.shape[0])
print("Input columns:", X.shape[1])
print("Missing values in X:", int(np.isnan(X).sum()))
print("Missing values in y:", int(np.isnan(y).sum()))

display(pd.Series(y, name="target").describe())

plt.figure(figsize=(9, 4))
plt.hist(y, bins=50)
plt.title("Distribution of supplied regression target")
plt.xlabel("Target value (unit not established)")
plt.ylabel("Record count")
plt.tight_layout()
plt.show()

## 5. Load and validate the official default split

Use the supplied indices unchanged. The inspected split contains 279,415 training, 34,174 validation and 36,927 test indices. The supplied indices reference rows 0–350,515; the remaining 65,935 rows are not assigned by this default split and must remain unused here.

In [ ]:
SPLIT_DIR = DATA_DIR / "splits" / "default"
train_idx = np.load(SPLIT_DIR / "train.npy")
val_idx = np.load(SPLIT_DIR / "val.npy")
test_idx = np.load(SPLIT_DIR / "test.npy")

n_records = len(y)
for split_name, indices in {
    "train": train_idx, "validation": val_idx, "test": test_idx
}.items():
    assert indices.ndim == 1, f"{split_name} indices must be one-dimensional"
    assert len(indices) > 0, f"{split_name} split is empty"
    assert indices.min() >= 0 and indices.max() < n_records

all_indices = np.concatenate([train_idx, val_idx, test_idx])
assert len(np.unique(all_indices)) == len(all_indices), "Split overlap detected"

print(f"Training:   {len(train_idx):,}")
print(f"Validation: {len(val_idx):,}")
print(f"Test:       {len(test_idx):,}")
print(f"Unassigned rows: {n_records - len(np.unique(all_indices)):,}")

## 6. Create the partitions

The test set is held back for final evaluation. Use validation results when comparing the two models.

In [ ]:
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

## 7. Define regression metrics

- **MAE:** average absolute prediction error.
- **MSE:** average squared prediction error; larger errors receive more weight.
- **RMSE:** square root of MSE; the documented benchmark metric.
- **R²:** fit relative to predicting the mean on the evaluated sample.

Metrics are on the supplied target scale. Do not label them as minutes, hours or another physical unit without authoritative target documentation.

In [ ]:
def evaluate_regression(model, X_part, y_part):
    """Calculate common regression metrics for one partition."""
    predictions = model.predict(X_part)
    mse = mean_squared_error(y_part, predictions)
    return {
        "MAE": mean_absolute_error(y_part, predictions),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_part, predictions),
    }


def show_metrics(model_name, metrics):
    print(f"\\n{model_name}")
    for metric_name, value in metrics.items():
        print(f"{metric_name:>6}: {value:,.6f}")

## 8. Train Linear Regression

The pipeline fits imputation and scaling using training data only. This prevents preprocessing statistics from being learned from validation or test records.

**Activity:** Run the cell and record the training and validation metrics.

In [ ]:
linear_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression()),
])

linear_model.fit(X_train, y_train)

linear_train_metrics = evaluate_regression(linear_model, X_train, y_train)
linear_val_metrics = evaluate_regression(linear_model, X_val, y_val)

show_metrics("Linear Regression — Training", linear_train_metrics)
show_metrics("Linear Regression — Validation", linear_val_metrics)

## 9. Train Ridge Regression

Ridge adds an L2 penalty to the linear regression objective. It can help control coefficient size when predictors are numerous or correlated.

`alpha=1.0` is a teaching baseline, not a claim of optimal tuning. Use validation data for model selection; do not tune against the test set.

**Activity:** Run the cell and compare validation RMSE.

In [ ]:
ridge_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regressor", Ridge(alpha=1.0)),
])

ridge_model.fit(X_train, y_train)

ridge_train_metrics = evaluate_regression(ridge_model, X_train, y_train)
ridge_val_metrics = evaluate_regression(ridge_model, X_val, y_val)

show_metrics("Ridge Regression — Training", ridge_train_metrics)
show_metrics("Ridge Regression — Validation", ridge_val_metrics)

## 10. Compare validation performance

Lower MAE, MSE and RMSE indicate smaller errors. Higher R² indicates a better fit relative to the mean baseline.

**Interpretation:** Which model has lower validation RMSE? Is the difference meaningful? What additional evidence would you need before recommending either model?

In [ ]:
comparison = pd.DataFrame({
    "Linear Regression": linear_val_metrics,
    "Ridge Regression": ridge_val_metrics,
}).T

display(comparison)
best_model_name = comparison["RMSE"].idxmin()
print("Lower validation RMSE:", best_model_name)

## 11. Evaluate on the held-out test partition

After model selection using validation data, test performance provides a final evaluation. For classroom comparison, this cell reports both models. In a production workflow, reserve test results for the final evaluation after model selection is complete.

In [ ]:
linear_test_metrics = evaluate_regression(linear_model, X_test, y_test)
ridge_test_metrics = evaluate_regression(ridge_model, X_test, y_test)

test_comparison = pd.DataFrame({
    "Linear Regression": linear_test_metrics,
    "Ridge Regression": ridge_test_metrics,
}).T

display(test_comparison)

## 12. Visualise predictions and residuals

The plots use up to 1,000 validation records for readability; the metrics above use the full validation partition.

\[
\text{Residual} = \text{Actual target} - \text{Predicted target}
\]

In [ ]:
selected_model = linear_model if best_model_name == "Linear Regression" else ridge_model

sample_size = min(1000, len(y_val))
actual_sample = y_val[:sample_size]
predicted_sample = selected_model.predict(X_val[:sample_size])
residual_sample = actual_sample - predicted_sample

plt.figure(figsize=(6, 6))
plt.scatter(actual_sample, predicted_sample, alpha=0.45)
lower = min(actual_sample.min(), predicted_sample.min())
upper = max(actual_sample.max(), predicted_sample.max())
plt.plot([lower, upper], [lower, upper], linestyle="--")
plt.xlabel("Actual target value")
plt.ylabel("Predicted target value")
plt.title(f"Actual vs Predicted — {best_model_name} (validation sample)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.scatter(predicted_sample, residual_sample, alpha=0.45)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted target value")
plt.ylabel("Residual")
plt.title(f"Residuals vs Predicted — {best_model_name} (validation sample)")
plt.tight_layout()
plt.show()

## 13. Business interpretation

Imagine an operations team wants to use a model to support delivery planning. Before recommending deployment, consider:

1. Does the model perform better on unseen records than a simple baseline?
2. Is validation performance reasonably close to training performance?
3. Are large errors concentrated in particular parts of the target range?
4. What is the business cost of over-prediction versus under-prediction?
5. Do we understand the target definition, feature meanings, data collection process and possible leakage risks?

**Limitation:** Current metadata confirms regression and RMSE scoring, but not the target's business unit or meaning. An operational recommendation requires authoritative target documentation and a data dictionary.

## 14. Hands-on challenge

### Task A — Compare models
- Compare validation RMSE for Linear Regression and Ridge Regression.
- State which model performs better on this validation partition.
- Explain whether the difference matters to the business.

### Task B — Investigate generalisation
- Compare training RMSE with validation RMSE for each model.
- Identify any gap and explain one possible reason.

### Task C — Change the Ridge penalty
- Change `alpha=1.0` to another value, such as `10.0`.
- Refit the Ridge pipeline and compare validation RMSE.
- Explain why a larger penalty may help or hurt.

### Task D — Communicate responsibly
Write a short recommendation to an operations manager. Include the model you would investigate further, validation evidence, one dataset-documentation limitation and one additional check required before deployment.

Do not tune against the test set.

## 15. Knowledge check

1. What type of target variable is regression designed to predict?
2. What is the practical difference between MAE and RMSE?
3. Why do we fit the imputer and scaler on training data only?
4. What does the Ridge `alpha` parameter control?
5. Why should the test set not be used repeatedly to choose model settings?

### Submission checklist
- [ ] Notebook runs from top to bottom.
- [ ] Official default split indices are used unchanged.
- [ ] Linear Regression and Ridge metrics are recorded.
- [ ] Validation comparison is completed.
- [ ] Training-versus-validation performance is discussed.
- [ ] Residual diagnostics are interpreted.
- [ ] Business recommendation includes limitations and next checks.

**Next session:** Day 5 — Classification.